# Загрузка, анализ, подготовка данных

## Задача

После разбора домашнего задания выяснелось, что данные были зашумлены. Вот некоторые из признаков проблемных переводов:
1) Длинна перевода больше длинны оргинала более чем в $n$ раз (и наоборот) Где $n$ определяется эмперически для каждого датасета;
2) В паре примера одно из полей (`src` и/или `dst`) пустое;
3) В переводе и оригинале используются арабские цифры, но они различаются;
4) в паре используются скобки и кавычки, они различаются.

## Подготовка среды

### Импорт библиотек

In [89]:
import random
import string
import re

import pandas as pd
import numpy as np
import plotly.express as px

from datasets import load_dataset, Features, Value, concatenate_datasets, Dataset, DatasetDict

### Полезные функции

In [90]:
# Функция для подсчета уникальных символов
def count_unique_characters_in_split(dataset, split_name, feature_name):
    
    unique_characters = set()
    
    for example in dataset[split_name][feature_name]:
        if example is not None:
            unique_characters.update(example)
    num_unique_characters = len(unique_characters)

    print(f"Number of unique characters in {split_name} {feature_name} split: {num_unique_characters}")
    print(f"Unique characters in {split_name} {feature_name} split: {unique_characters}", end="\n\n")

    return num_unique_characters, unique_characters

# Функция для подсчета отношения длин src и dst и рисования гистограммы
def plot_length_difference_distribution(df, split_name='train', bins=100):
    df['length_difference'] = df['src'].apply(len) / df['dst'].apply(len)
    fig = px.histogram(df, x='length_difference', nbins=bins, title=f'Распределение разницы в длине между src и dst для {split_name}')
    
    tick_step = 0.25
    x_range = [0, 3.5]
    tickvals = np.arange(x_range[0], x_range[1] + tick_step, tick_step)
    
    fig.update_xaxes(title='Отношение длин', range=x_range, tickvals=tickvals)

    fig.show()

## Загрузка данных

Загрузим данные и посмотрим на случайно выбранный примеры из каждой части датасета:

In [91]:
data_files = {
    "train": "raw_train.jsonl",
    "validation": "raw_val.jsonl",
    "test": "raw_test_no_reference.jsonl"
}

features = Features({
    'src': Value('string'),
    'dst': Value('string'),
})

dataset = load_dataset("json", data_files=data_files, features=features)
dataset

DatasetDict({
    train: Dataset({
        features: ['src', 'dst'],
        num_rows: 300000
    })
    validation: Dataset({
        features: ['src', 'dst'],
        num_rows: 500
    })
    test: Dataset({
        features: ['src', 'dst'],
        num_rows: 1000
    })
})

In [92]:
example_index = random.randint(0, dataset['validation'].shape[0] - 1)

print(f"Train example: {dataset['train'][example_index]}, src lengh {len(dataset['train'][example_index]['src'])}, dst lengh {len(dataset['train'][example_index]['dst'])}")
print(f"Validation example: {dataset['validation'][example_index]}, src lengh {len(dataset['validation'][example_index]['src'])}, dst lengh {len(dataset['validation'][example_index]['dst'])}")
print(f"Test example: {dataset['test'][example_index]}, src lengh {len(dataset['test'][example_index]['src'])}")

Train example: {'src': '◫▱◪ ◀▴◓◠◀◪◓ ◉◠▱▪◒◎◠◳◠ ◈◠ ◈▴◚◠◎ ◪◈▴▼◪▨◞◗▦▵', 'dst': 'and still do everything you have to for Harvey.'}, src lengh 41, dst lengh 47
Validation example: {'src': '◄▾◠▱▱◗◎■ 2011\'◈▴▦ ◀◨ ◳◠▦◠ 360▵000\'◈▴▦ ◍◠▢▱◠ ◫▦◞◠▦◬▦ ◇▱◈▩◐▩ ◚◪ ◎◫▱▽◧▦▱◠◓▼◠◞◬▦▪▦ ◈◠▷◠ ◪◚◞◫▢ ▨◠▱◈▪◐▪ ▮▾◓◫▽◪\'◈▴ "▫▴◓◣◓▱▴ ◞◠◚◠◒▪▦ ▦▴◓▴◈▴◳◞▴ ◀◫▫▫◗◐◗▦◫" ◫◈◈◫◠ ◪▫▫◗▵', 'dst': 'Moualem insists that in Syria, where more than 360,000 people have died since 2011 and millions have been displaced from their homes, "the war on terrorism is almost over."'}, src lengh 157, dst lengh 172
Test example: {'src': '▭◠▨▱▪ ◀◫◓ ▴▱◪◒▫◗◓◗■ ◠◳▱◠◓▪▦ ◚▴ ▽▪▱▱◠◓▪▦ ◈◂▱◨▻ ▫◠◒◠▦ ◇◍▨◪◞◫ ◈◪ ◂▱◞◠ ◀◨▦▱◠◓▪ ◠◐▱◠◎◠◈◠▦ ◈◗▱◪ ◕◪▫◗◓◪◎◗◳◧◓▵', 'dst': None}, src lengh 102


Судя по выводу, данные загрузились без проблем с кодировкой.

## Очистка данных

### Мусорные символы

Получим множества символов используемых в обучающей, валидационной и тестовой выборках признаков `src` и `dst` раздельно. Найдем загрязняющие символы для обоих признаков:

In [94]:
# Признак src
train_src_unique_char_count, train_src_unique_chars = count_unique_characters_in_split(dataset, 'train', 'src')
validation_src_unique_char_count, validation_src_unique_chars = count_unique_characters_in_split(dataset, 'validation', 'src')
test_unique_src_char_count, test_src_unique_chars = count_unique_characters_in_split(dataset, 'test', 'src')

# Признак dst
train_dst_unique_char_count, train_dst_unique_chars = count_unique_characters_in_split(dataset, 'train', 'dst')
validation_dst_unique_char_count, validation_dst_unique_chars = count_unique_characters_in_split(dataset, 'validation', 'dst')

# Загрязняющие символы для каждого из признаков
src_chars_to_remove = train_src_unique_chars - (validation_src_unique_chars | test_src_unique_chars)
print(f"SRC feature have {len(src_chars_to_remove)} symbols for remove: {src_chars_to_remove} )")

dst_chars_to_remove = train_dst_unique_chars - validation_dst_unique_chars
print(f"DST feature have {len(dst_chars_to_remove)} symbols for remove: {dst_chars_to_remove} )")

Number of unique characters in train src split: 176
Unique characters in train src split: {'1', '£', '◳', '▦', '►', '▷', '2', '%', 'þ', '◕', 'û', '◭', '#', '◘', '◈', '▣', 'â', '▫', '=', '◢', '◌', '\x99', 'ñ', '*', '◖', ']', '▿', '—', '~', 'è', 'ó', 'ο', 'Ä', '△', '▧', '◦', '5', '◩', ':', '▥', 'ø', '▻', 'Î', '¶', 'Â', '\xa0', '◬', '◙', '◃', '◰', '4', 'ß', 'á', '[', '◅', 'ï', 'í', '◔', '_', '◛', '◝', '◓', '\u202d', '◁', '"', '▱', '♫', 'º', '0', '◀', '/', '§', '◊', '◲', '◞', '◠', '◡', '°', '▩', '‚', '!', '◄', '•', 'Ý', '6', '○', '◇', '─', '\\', '}', '◣', '◜', '–', 'î', '◑', '▼', '◥', '™', '▤', 'ä', '◎', '▪', 'Þ', '◧', '$', '8', '◫', '◉', '3', '▽', '`', '▢', '¿', 'ι', '◱', 'ý', '◨', '◚', '▰', '{', '♪', ' ', '◯', '◤', '7', 'ô', '◍', '◒', 'ð', 'đ', '¤', '◪', '◐', 'ν', '(', 'ţ', 'ă', '▯', '▨', '▬', '@', '▵', 'é', '●', '◗', '▮', 'å', 'ë', '\x9d', '¡', '?', '\x9e', '^', '\u200b', '□', '▶', 'É', '▲', '▴', '▭', '◮', '´', '▸', '+', '■', '◂', '9', '◟', '◆', '▹', 'ú', ')', '▾', ';', 'ª', "'"}

Numbe

К тренировочной части датастета небоходимо применяться одно и тоже преобразование, что применялось к тестовой и валидационной части. Удалим из тренировочной части датасета все найденные загрязняющие символы за исключением символов английского алфавита, арабских цифр и символов пунктуации: 



In [100]:
chars_to_remove = (src_chars_to_remove | dst_chars_to_remove | {'½', 'á'} ) - set(string.ascii_letters + string.digits) - {'.', ',', '!', '?', '—', '–', '[', ']', '$', '£', '&', '\\', '#', '”', '“'} 

print(f"Total {len(chars_to_remove)} symbols for remove: {chars_to_remove}")

Total 122 symbols for remove: {'Ż', 'þ', 'Μ', 'û', 'â', 'с', '=', 'Ð', '±', '\x99', 'ñ', 'ć', '*', 'Ë', '\x8d', 'ﬂ', 'υ', 'Ν', '~', 'è', 'ó', 'Τ', 'œ', 'ο', 'Ä', 'š', '€', '©', 'ø', 'Æ', 'Î', '¶', 'Á', 'Â', 'ē', '\xa0', '‒', 'ß', 'á', 'ï', '½', 'í', '_', 'Ü', '\u202d', '♫', 'º', '§', 'à', '°', '‚', 'Ο', '•', 'Ý', '─', '}', 'î', 'İ', '™', 'ş', 'Ã', 'ä', '\x97', 'ﬁ', '\xad', 'Þ', 'ò', 'ç', '`', '¿', '檛', 'ι', '″', 'ý', '³', '¯', 'æ', 'ρ', '{', '\x92', 'ư', '♪', 'Ι', 'ô', 'ð', 'đ', '\x90', '¤', 'ν', 'ı', 'ţ', 'ă', 'ì', 'õ', 'у', '│', '●', '¬', 'å', 'ë', '\x9d', '¡', 'É', '\x9e', '^', '\u200b', '¢', 'Ö', 'ö', 'Β', 'Η', '˜', 'ã', '´', 'À', '鈥', '′', '\x83', 'ú', 'ί', 'ª', 'ê'}


In [ ]:
# # Изучение символов датасета.
# symbol = r'“'

# df_test = dataset['test'].to_pandas()
# df_validation = dataset['validation'].to_pandas()


# filtered_validation_df = df_validation[df_validation['src'].str.contains(symbol) | df_validation['dst'].str.contains(symbol, regex=True)]
# filtered_test_df = df_test[df_test['src'].str.contains(symbol, regex=True)]

# filtered_validation_df

Удалим из тренировочной части датасета все найденые грязные символы:

In [101]:
# def remove_chars_from_text(text, chars_to_remove):
#     for char in chars_to_remove:
#         text = text.replace(char, '')
#     return text

# def clean_dataset(dataset, chars_to_remove):
#     def clean_example(example):
#         example['src'] = remove_chars_from_text(example['src'], chars_to_remove)
#         example['dst'] = remove_chars_from_text(example['dst'], chars_to_remove)
#         return example

#     dataset['train'] = dataset['train'].map(clean_example, num_proc=10)
#     return dataset

# dataset = clean_dataset(dataset, chars_to_remove)

def clear_example(example):
    for char in chars_to_remove:
        for feature in example:
            example[feature] = example[feature].replace(char, '') 

    return example

dataset['train'] = dataset['train'].map(clear_example, num_proc=10, load_from_cache_file=False)

Map (num_proc=10):   0%|          | 0/300000 [00:00<?, ? examples/s]

Проверим сколько теперь символов в частях датасета:

In [102]:
# Признак src
train_src_unique_char_count, train_src_unique_chars = count_unique_characters_in_split(dataset, 'train', 'src')
validation_src_unique_char_count, validation_src_unique_chars = count_unique_characters_in_split(dataset, 'validation', 'src')
test_unique_src_char_count, test_src_unique_chars = count_unique_characters_in_split(dataset, 'test', 'src')

print(f"Difference on src: {test_src_unique_chars  - (validation_src_unique_chars | train_src_unique_chars)}", end="\n\n")

# Признак dst
train_dst_unique_char_count, train_dst_unique_chars = count_unique_characters_in_split(dataset, 'train', 'dst')
validation_dst_unique_char_count, validation_dst_unique_chars = count_unique_characters_in_split(dataset, 'validation', 'dst')


print(f"Difference on dst: {validation_dst_unique_chars - train_dst_unique_chars}", end="\n\n")

Number of unique characters in train src split: 115
Unique characters in train src split: {'1', '£', '◳', '▦', '►', '▷', '2', '%', '◕', '◭', '#', '◘', '◈', '▣', '▫', '◢', '◌', '◖', ']', '▿', '—', '△', '▧', '◦', '5', '◩', ':', '▥', '▻', '◬', '◙', '◃', '◰', '4', '[', '◅', '◔', '◛', '◝', '◓', '◁', '"', '▱', '0', '◀', '/', '◊', '◲', '◞', '◠', '◡', '▩', '!', '◄', '6', '○', '◇', '\\', '◣', '◜', '–', '◑', '▼', '◥', '▤', '◎', '▪', '◧', '$', '8', '◫', '◉', '3', '▽', '▢', '◱', '◨', '◚', '▰', ' ', '◯', '◤', '7', '◍', '◒', '◪', '◐', '(', '▯', '▨', '▬', '@', '▵', 'é', '◗', '▮', '?', '□', '▶', '▲', '▴', '▭', '◮', '▸', '+', '■', '◂', '9', '◟', '◆', '▹', ')', '▾', ';', "'"}

Number of unique characters in validation src split: 116
Unique characters in validation src split: {'1', '◳', '▦', '►', '▷', '2', '%', '◕', '◭', '▣', '◘', '◈', 'â', '▫', '◢', '◌', '◖', ']', '▿', '△', '▧', '◦', '5', '◩', ':', '▥', '▻', '“', '◬', '◙', '◃', '◰', '4', '[', '◅', 'í', '◔', '◛', '◝', '◓', '"', '◁', '▱', '0', '◀', '/', '

Преобразуем обучающую часть датасета в Dataframe, для более удобной работы:

In [138]:
df_train = pd.DataFrame(dataset['train'])
df_validation = pd.DataFrame(dataset['validation'])
df_test = pd.DataFrame(dataset['test'])
df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,Grasshopper is moving out of the crowd.
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Строки в которых нет английских букв в признаке `dst`

In [139]:
# Фильтрация строк, в которых в признаке dst нет английских букв
df_train_no_english = df_train[~df_train['dst'].str.contains('[a-zA-Z]', regex=True)]
df_train_no_english

,src,dst
1366,◝◨▦◨ ◞◠▦◠ ▨◗◎ ◞◇◳▱◪◈◗?,
1739,"""◟○▲○▯▬◃ ◙◘◙◰◢▶◲◌""",
2257,► ◡◠▻◧▦ ◎◠▱◬▵,"- $130,000."
3601,◤◗◓◎◗ ◀▴◒▵,285...
5646,◂▱◈▾◐▾▦◨ ◈◭◒◭▦◎▩◒▫▩◎▵ ○◎◠ ◈◪◐◗▱◈◫■ ◀▩▽◭▨ ◀◫◓ ◒...,
...,...,...
294982,◝◗◓◕◪◓■ ▷▴◓ ▢◠◎◠▦ ◠◀◠◓▫▪▱▪ ◈◠◚◓◠▦◬▽◧◓ ▢◠▫◪▦▵,
295836,# ◆◇▦◈▴◓ ▫◂▻▱◨▱◨◐◨▦ ◗◉◗▦▴ #,
296309,▮◪▦◫ ▫◪◀◓◫▨ ▴▫◎◪◳◗ ◈◭◒◭▦▩◳◧◓◈◨◎ ◠◎◠ ◀▾ ◕◭◚◪▦◗▦...,
299240,1400▵ 1400 ◈◂▱◠◓ ◚◪◓◪▦ ◚◠◓ ◎◬? 1400▵,"1,400."


Удалим такие примеры:

In [140]:
df_train = df_train.drop(df_train_no_english.index)
df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,Grasshopper is moving out of the crowd.
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Вопросительные и восклицательные предложения предложения

Знаки `!` и `?` в обоих языках должны совпадать в конце предложения. Найдём примеры в которых это не соблюдается:

In [141]:
examples_with_exclamation_question_mismatch = df_train[(df_train['src'].str.endswith(('!', '?'))) & ~df_train['dst'].str.endswith(('!', '?'))]
examples_with_exclamation_question_mismatch

,src,dst
69,◢◠▻◠◐◬▦ ◠◉▪▱◎◠◞▪ ◫▱◕◫▦◗ ◉▴▨◎▴◈◫ ◎◫?,No interest in the hatch blowing.
146,◝◗▢◫ ◀▾◓◠▽◠ ▦◪◈▴▦ ◕◪▫◗◓◈◫▦?,There are trash cans.
185,▶◠◎◠◎■ ◚◠◳ ▼◠▦◬▦◠!,Ok. Wow.
228,◝◪▦◫◎▱▴ ◠▱◠▽ ▴◈◗◳◧◓◞◨▦ ▷▴◓▷▱◈▴?,You've got to be kidding me.
240,◑◪◓ ◒◨▦▾■ ◳◭◓◭ ▷◠◈◗■ ▷◠◓◪▨▴▫ ▴▫!,"Give me that. Go on, get moving."
...,...,...
299811,◤◠▫▫▪◐◬◎▪▢◈◠■ ◀◪▦ ◗▱▨ ◎◗ ◧▱◠▼◠◐▪◎?,"When we bed together, shall I be your first? -..."
299855,◆◣◓▴◚◫ ◀▪◓◠▨◎▪◒ ◂▱◨◓◈◨▨?,We would've dropped the charges.
299879,▭◠▽◈◫ ◂◐▱▾◎!,"Come on, boy."
299980,► ◄◫▨◪ ▶▽◞◧▦'▪ ▨◫◎ ◉◠▱▪◒▫◬◓◈◬ ◞◠▦▪◳◂◓◞▾▦?,- Yeah. - Gus D'Amato.


Удалим такие примеры из тренировочной части датасета:

In [142]:
df_train = df_train.drop(examples_with_exclamation_question_mismatch.index)
df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,Grasshopper is moving out of the crowd.
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


###  Примеры заканчивающиеся символами `:` в обоих признаках

Посмотрим на примеры которые заканчиваются символом `:` в признаках `src` и не заканчиваются этим символом в признаке `dst` и наоборот:

In [159]:
# Examples where src ends with ':' but dst does not
examples_src = df_train[(df_train['src'].str.endswith(':')) & (~df_train['dst'].str.endswith(':'))]

# Examples where dst ends with ':' but src does not
examples_dst = df_train[(~df_train['src'].str.endswith(':')) & (df_train['src'].str.endswith(':'))]

Удалим из тренировочной части такие примеры:

In [160]:
df_train = df_train.drop(examples_src.index)
df_train = df_train.drop(examples_dst.index)

df_train

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
1,▽◪◎◗▦◫▦◫ ▫▴▨◓◠◓ ▴▫◎◪▱◫ ◚▴ ◞◧▦◞▾▢▱◨▨ ◒◠◓◠◀▪▦◈◠▦...,He would need to repeat his vows in the land o...
2,◄▴◞◠▸▱◠◓▪◎◠ ◀◫▱◪ ▼◪◚◠▻ ◚▴◓▴◎◪◈◗▦ ◎◫?,You couldn't even answer my texts?
3,▯◪ ▨◠◈◠◓ ◞◭◓◠▫ ◳◠▻◬◳◧◓ ◞▴▦◗▦▨◫?,How fast do you go?
4,◈◠ ◧▱◠▦ ◀◫◓ ▨◠◉ ◂▱◠▽◈◠▦ ◀◠▷◞◪◈◗◳◧◓■ ◉◧◐▾▦▱◨◐▾ ...,"He's talking about a few right here in Lisbon,..."
...,...,...
299995,►◅▴▨◫◓◕▴ ▨◠▱◠◀◠▱◬◐▪▦ ◈▪◒◬▦◠ ◈◧◐◓▾ ▷◠◓▴▨▴▫ ▴◈◗◳◂◓▵,Grasshopper is moving out of the crowd.
299996,◝◓◳▼▴ ◉◂▼▾◐▾◎◠ ◫◳◫ ◀◠▨,"Brice, I need you take care of my child. I wil..."
299997,◅◂▨ ◒◗◓◫▦ ◀◗◓ ◉◂▼◨▨▫◨▵,He was such a sweet kid.
299998,◮▢◨▦ ▢◠◎◠▦◈▪◓ ◨▢◠▽◈◠ ◈▴◐◗▱◈◗◎▵,I haven't been in space for a long time.


### Примеры с проблемами с прямой речью

В тестовой и валидационной частях датасета нет примеров заканчивающихся на символ 

In [162]:
examples_with_quote = df_validation[df_validation['src'].str.endswith('"') & df_validation['dst'].str.endswith('"')]
examples_with_quote

pandas.core.frame.DataFrame

In [163]:
examples_with_quote = df_test[df_test['src'].str.endswith('"')]
examples_with_quote

,src,dst
16,"""◢◠▦▫◪'▦◫▦ ◕◪▱◈◗◐◫▦◗ ◕◣◓◭▻ ◀◫◓ ▨▴◓▴ ◈◂▨▾▦◈◨◎▵ ...",None
27,"◄◂◓◠▱▴◞'▴ ◕◇◓▴ ""◝◧▱◗◚◳◠ ◠◞▱◠ □◠◞◫◍◫▨ ◱▨◳◠▦◨◞▾'...",None
42,▮◠▦◈◪◓◞ ◀◓◫◍◗▦◕▱◪◓◗▦ ◠▢ ◧▱◎◠◞▪▦◬▦ ▦◪◈◪▦◫▦◗▦ ▶◑...,None
52,"○▱◀◭◎ ◈◠▷◠ ◉◂▨ ▨◫◎ ◧▱◈◨◐▾◎◨▢▾▦ ◀◗◓ ◳◠▦◞▪◎◠◞◬▵""",None
62,▰◪◞▫ ◑◫◓◕◗▦◗◠ ◄◪▫◓◂ ▯▴▥◞'▴ ◕◣◓▴■ ◀▾ ◠▨◒◠◎ ▰▷◪◪...,None
...,...,...
950,"◤◠▻◎◠▽◬▦▵""",None
953,◝◠▨◠▦ ◀◨▦▾▦ ▽▴◓◫▦◪ ◞◂▦ ◀▴◒ ◠◳◈◠ ◢◫◎ ◗▱◪ ◆◭▦▴▽ ...,None
965,2018 ▫◠▨◬◎▪▦◬▦ ▲▴ ◆◧▱◍ ▯◠▫◗◂▦◠▱'◈◪ ▷◪◎ ▬◨◎◠ ◕▩...,None
983,◝▾ ◒◫◎◈◗◈▴▦ ◀◭◳▩▨ ◀◗◓ ▫◓◠▸◪◈◗ ◠◎◠ ◈◨◓◨◎ ◈◠▷◠ ◈...,None


### Прямая речь в предложениях

Найдём все примеры из тренировочной части датасета, где признак dst начинается с символа '-' или '—':

In [108]:
examples_with_dash = df_train[df_train['dst'].str.startswith(('-', '—'))]
examples_with_dash

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
22,◝▾ ◫◒◫ ◀◫◓ ◒◪▨◫▱◈◪▵▵▵ ► ▭◗◉ ◒◠▦◞◬▦ ◳◧▨▵,- We gotta work something...
36,◆◇▫◭◓◪▼◪◐◫◎■ ◝◪◀◠▼◬◐◬◎▵,"- Of course, Beba."
43,► ◝◠▦◠ ◳◠◓◈◬◎ ◪▫◎◪▦◗ ◗◞▫▴◎◫▽◂◓◨◎!,- I don't want you to help me!
47,► ◢◇◒◪◳◫ ◈◣▦▩▦▼▴ ◕◠▱◗◀◠▵,"- (Irene moans) - (Johnny) Come on, y'all."
...,...,...
299976,► ▭◠◈◫ ◞▴▦◗ ▫▴◎◗▢▱◪◳▴▱◗◎▵,- Let's get you cleaned up.
299979,► ◙◑◙ ▽◠▦▪◎◈◠▵,- I've got the DVD here.
299985,► ◢◠◀▾▱ ▴◈▴◀◗▱◪▼◪◐◫▦◈▴▦ ◉◂▨ ◞▾◉▱◨◞◨▦▵,- You're more to blame than you'll admit.
299987,◌◓◠▦▨■ ◉▪▨ ◧◓▫◠▽◠▵,"- Jimmy crack corn... - Frank, come on out!"


Проверим, сколько примеров в тренировочной части датасета не начинаются с символа `►` в признаке `src` но начинаются с символов '-' или '—' в признаке `dst`:

In [110]:
examples_with_dash_mismatch = df_train[(~df_train['src'].str.startswith('►')) & (df_train['dst'].str.startswith(('-', '—')))]
examples_with_dash_mismatch

,src,dst
0,◄▴◓◠▨ ◨▽◠▦◈◬◓▪▼◬▵,- Intriguing.
22,◝▾ ◫◒◫ ◀◫◓ ◒◪▨◫▱◈◪▵▵▵ ► ▭◗◉ ◒◠▦◞◬▦ ◳◧▨▵,- We gotta work something...
36,◆◇▫◭◓◪▼◪◐◫◎■ ◝◪◀◠▼◬◐◬◎▵,"- Of course, Beba."
94,◢◠▻◠ ◉◪▦◪▦◫ ◈▴ ◉▪▨◠◓ ▩◞▫◭▦◈▴▨◫▦◗▵,- Shut up and get outta your gear. - I'm takin...
152,◝◨◓◠◈◠ ▫▴▨ ◀◠◒▪◎▪▢◠ ◂▱◎◠◎▪▢ ◀▴▦◫ ◓◠▷◠▫◞▪▢ ▴◈◫▽◧◓▵,"- I'm not comfortable staying here, just us."
...,...,...
299889,◆◠◫▱ ◳◠▫▫▪ ◎◬?,- Did Gail go to bed?
299898,▿◍◫◞◫◎◗ ◎◗ ◨▦▾▫▫◨▦■ ▬◂◂▻? ◩◐▱◪ ◳▴◎▴▨▱▴◓◗▦◈▴ ◞◗...,- and books you sneak in to look at during lunch?
299972,◝◭◳▩▨ ◫▷▫◫◎◠▱▱▴ ◀▩▫▩▦ ◕▴▼◪ ▾▽◠▦▪▨▫▪▵▵▵,-He was probably up all night dicking that girl.
299975,◩◐◓▴▫◪▼▴◐◫▢■ ◎▴◓◠▨ ◪▫◎◪▵,"- We will, my dear, we will."


In [116]:
examples_with_dash_reverse = df_train[(df_train['src'].str.startswith(('-', '—'))) & (~df_train['dst'].str.startswith('►'))]
examples_with_dash_reverse

,src,dst
15914,— □◗▻▴◓ ▦◠◞◬▱?,- How's Piper?
20383,— ◊◳◫ ◒◠▦◞▱◠◓▵,- Good luck.
25087,— ◆◫◓◪◀◫▱◫◓ ◎◫▽◫◎? — ▿▱◨◓▵,- May I come in?
29058,— ◲◒▴ ◳◠◓◠◎◠▢▵▵▵,- It doesn't work...
37116,— ◄◠◓◫▷▾◠▦◠ ◈◠▷◠ ◀◗▢◪ ◕▴▱◎◪◎◫◒▫◗ ◀◫▱▴▵,That marihuana never even made it to us. - I w...
51177,— ▭◪◎◪▦ ◈◇▦▴◓◗◎■ ▲▴◂▵,"- Be right back, Leo."
52430,— ◢◗◎◞▴▦◫▦ ◀▴▦◗ ◞▴◚◎◪◞◗▦◗ ◗◞▫◪◎◫▽◂◓▾◎▵,- I don't want anybody to love me.
54958,— ▯◪ ◈▴◎◪▨ ◀◨ ◒◗◎◈◫? — ▯▴ ▢◠◎◠▦ ◀◫◓ ◒▴▽▵▵▵,- Every time I try to get something...
55305,— ▮◂◓▾▦ ◈▴◐◫▱ ◆◧◓◠▨▵,"- No problem, Gorak."
63919,— ▮◗▢◗ ▩▢◎◪▨ ◫◞▫▴◎◪▢◈◫▨■ ◠◎◠ ◀▾ ◞◬▨ ◞▪▨ ◧▱▾◓▵,"We don't want to upset you, but it happens."


Оставим только примеры в которых `src` начинается с символа '►' и `dst` начинается с символов '-', '—': 

In [117]:
df_train = df_train[(df_train['src'].str.startswith('►')) & (df_train['dst'].str.startswith(('-', '—')))]
df_train

,src,dst
43,► ◝◠▦◠ ◳◠◓◈◬◎ ◪▫◎◪▦◗ ◗◞▫▴◎◫▽◂◓◨◎!,- I don't want you to help me!
47,► ◢◇◒◪◳◫ ◈◣▦▩▦▼▴ ◕◠▱◗◀◠▵,"- (Irene moans) - (Johnny) Come on, y'all."
55,► ◊◳◗ ▽◂▱▼▾▱◨▨▱◠◓▵,"- Have a ""bon voyage"""
56,► ◊◳◗ ▽◂▱▼▾▱◨▨▱◠◓▵,-Have a great trip.
113,► ▭◗▦◈◗ ▻◗◒◗◓▴◀◫▱◗◓ ◎◗◳◫◎?,- Can I make a turkey?
...,...,...
299973,►▿▦◠ ◉◧▨ ◫▽◫ ◀◠▨◬▦▵,- Give him the best of care.
299976,► ▭◠◈◫ ◞▴▦◗ ▫▴◎◗▢▱◪◳▴▱◗◎▵,- Let's get you cleaned up.
299979,► ◙◑◙ ▽◠▦▪◎◈◠▵,- I've got the DVD here.
299985,► ◢◠◀▾▱ ▴◈▴◀◗▱◪▼◪◐◫▦◈▴▦ ◉◂▨ ◞▾◉▱◨◞◨▦▵,- You're more to blame than you'll admit.


### Количество слов предложении

Посчитаем количество слов в каждом примере для признаков `src` и `dst` для каждого примера в валидационной части датасета. Визуализируем полученное распределение:

### Примеры с большой разницей в длинне

Посмотрим, во сколько раз отличаются по длинне признаки примеров `src` и `dst` в тренировочной и валидационной частях датасета:

In [ ]:
plot_length_difference_distribution(df_train, 'train', bins=300)

In [ ]:
plot_length_difference_distribution(df_validation, 'validation', bins=300)

Хорошо видно, что большинство примеров валидации укладывается в диапазон от 0.5 до 1.75 включительно. Что говорит о том, что наш язык символьный и в тренировочной части датасета ~ 20k примеров можно выкинуть. Удалим все примеры соотношение длин признаков которых менее 0.5 и более 1.75:

In [ ]:

df_train = df_train[(df_train['length_difference'] > 0.7) & (df_train['length_difference'] < 1.3)]
df_train = df_train.drop(columns=['length_difference'])

df_train

### Примеры в которых различается порядок и количество цифр

Проверим наличие примеров в которых различается количество и порядок цифр:

In [ ]:
def extract_digits(text):
    return ''.join(re.findall(r'\d', text))

df_train['src_digits'] = df_train['src'].apply(extract_digits)
df_train['dst_digits'] = df_train['dst'].apply(extract_digits)

digit_mismatch_df = df_train[df_train['src_digits'] != df_train['dst_digits']]

digit_mismatch_df

Удалим найденые битые примеры:

In [ ]:
# Оставим только строки в которых количество и порядок цифр в src и dst совпадает
df_train= df_train[df_train['src_digits'] == df_train['dst_digits']]

# Удалим ненужные признаки
df_train = df_train.drop(columns=['src_digits', 'dst_digits'], errors='ignore')


df_train

### Проблемы с прямой речью

Посмотрим на примеры которые начинаются с символа `►` в `src` и не начинаются с символа `-` или `—` в `dst`:

In [ ]:
filtered_examples = df_train[(df_train['src'].str.startswith('►')) & (~df_train['dst'].str.startswith(('-', '—')))]
filtered_examples

Это примеры с поврежденной прямой речью, удалим их:

In [ ]:
df_train = df_train[~((df_train['src'].str.startswith('►')) & (~df_train['dst'].str.startswith(('-', '—'))))]
df_train

Посмотрим на обратную ситуацию, когда в `dst` есть символы `-` или `—`, но в `src` нет символа `►` в начале строки:  

In [ ]:
filtered_examples_reverse = df_train[(df_train['dst'].str.startswith(('-', '—'))) & (~df_train['src'].str.startswith('►'))]
filtered_examples_reverse

Судя по всему это тоже битая прямая речь, удалим ее:

In [ ]:
df_train = df_train[~((df_train['dst'].str.startswith(('-', '—'))) & (~df_train['src'].str.startswith('►')))]
df_train

### Примеры в которых различаются символы пунктуации

Проверим наличие примеров в которых в признаках различаются символы пунктуации:

In [ ]:
## ЗАГЛУШКА

### Дедубликация примеров

Проверим наличие дубликатов в примерах по признаку `src`:

In [ ]:
duplicates = df_train[df_train.duplicated(subset=['src'], keep=False)]
duplicates

Удалим все дубликаты оставляя первый встреченный пример:

In [ ]:
df_train = df_train.drop_duplicates(subset=['src'], keep='first')
df_train

Проверим дубликаты по признаку `dst`:

In [ ]:
# duplicates = df_train[df_train.duplicated(subset=['dst'], keep=False)]
# duplicates

## Сохрание очищеной тренировочной части датасета

In [ ]:
# Обновление датасета после фильтрации
dataset['train'] = Dataset.from_pandas(df_train, preserve_index=False)

# Проверка количества строк после фильтрации
print(f"Number of rows after filtering in train split: {dataset['train'].shape[0]}")


dataset['train'].to_json('train.jsonl', orient='records', lines=True, index=False, force_ascii=False)